In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta, date

df = pd.read_csv("datasets/BE_Data_UTC.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["date_only"] = df["Date"].dt.date
df["hour"] = df["Date"].dt.hour
df = df.sort_values("Date").reset_index(drop=True)

def naive_forecast(forecast_date, df):
    if forecast_date.weekday() in (1, 2, 3, 4):
        lag_days = 1
    else:
        lag_days = 7
    source_date = forecast_date - timedelta(days=lag_days)
    source_rows = df[df["date_only"] == source_date].sort_values("hour")
    if len(source_rows) != 24:
        return None, None
    return source_rows["Price"].values, source_date

forecast, source_date = naive_forecast(date(2023, 6, 1), df)
print(forecast[:5])
print(source_date)

[74.2  62.64 61.61 72.7  77.64]
2023-05-31


In [3]:
import pandas as pd
from datetime import timedelta, date

df = pd.read_csv("datasets/BE_Data_UTC.csv")
df["Date"] = pd.to_datetime(df["Date"])
df["date_only"] = df["Date"].dt.date

test_date = date(2023, 6, 1)  # any date you're testing in the dashboard
day = df[df["date_only"] == test_date].sort_values("Date")
print(day[["Price", "Price_CH", "Price_DE_LU_15min", "Price_AT_15min"]].head())

       Price  Price_CH  Price_DE_LU_15min  Price_AT_15min
21888  70.10     69.60             72.885         66.1800
21889  65.05     66.49             71.535         67.0200
21890  67.40     67.89             71.400         67.8775
21891  75.39     74.55             82.015         78.1025
21892  88.28     86.15             96.495         91.8275


In [1]:
import sqlite3

# 1. Connect (creates the file if it doesn't exist yet)
conn = sqlite3.connect("test_practice.db")

# 2. Create a table
conn.execute("""
    CREATE TABLE IF NOT EXISTS notes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT NOT NULL
    )
""")
conn.commit()

In [2]:
conn.execute("INSERT INTO notes (content) VALUES (?)", ("my first note",))
conn.execute("INSERT INTO notes (content) VALUES (?)", ("second note",))
conn.commit()

In [3]:
cursor = conn.execute("SELECT * FROM notes")
rows = cursor.fetchall()
print(rows)

[(1, 'my first note'), (2, 'second note')]


In [4]:
cursor = conn.execute("SELECT * FROM notes WHERE id = ?", (1,))
print(cursor.fetchone())  # fetchone() instead of fetchall() when you expect at most one row

(1, 'my first note')


In [5]:
conn.execute("UPDATE notes SET content = ? WHERE id = ?", ("edited note", 1))
conn.execute("DELETE FROM notes WHERE id = ?", (2,))
conn.commit()

In [6]:
conn.close()

In [15]:
import sys
# sys.path.append(".")  # only needed if the notebook isn't already in the same folder as db.py

from db import load_feedback, load_users

feedback_df = load_feedback()
feedback_df.head(20)

,id,expert_id,forecast_date,timestamp_slot,forecast,adjusted,flagged,load_fr,confidence,timestamp
0,1,expert_1,2026-08-14,2026-08-14 00:00:00,140.824265,140.824265,False,39600.0,4,2026-08-13T23:49:56.955283+00:00
1,2,expert_1,2026-08-14,2026-08-14 00:15:00,136.857758,136.857758,False,38732.0,4,2026-08-13T23:49:56.955283+00:00
2,3,expert_1,2026-08-14,2026-08-14 00:30:00,134.259003,134.259003,False,37864.0,4,2026-08-13T23:49:56.955283+00:00
3,4,expert_1,2026-08-14,2026-08-14 00:45:00,127.040955,127.040955,False,37451.0,4,2026-08-13T23:49:56.955283+00:00
4,5,expert_1,2026-08-14,2026-08-14 01:00:00,132.072113,132.072113,False,37037.0,4,2026-08-13T23:49:56.955283+00:00
5,6,expert_1,2026-08-14,2026-08-14 01:15:00,128.889099,128.889099,False,36784.0,4,2026-08-13T23:49:56.955283+00:00
6,7,expert_1,2026-08-14,2026-08-14 01:30:00,127.641571,127.641571,False,36530.0,4,2026-08-13T23:49:56.955283+00:00
7,8,expert_1,2026-08-14,2026-08-14 01:45:00,123.875687,123.875687,False,36279.0,4,2026-08-13T23:49:56.955283+00:00
8,9,expert_1,2026-08-14,2026-08-14 02:00:00,127.257126,127.257126,False,36028.0,4,2026-08-13T23:49:56.955283+00:00
9,10,expert_1,2026-08-14,2026-08-14 02:15:00,125.198006,125.198006,False,35955.0,4,2026-08-13T23:49:56.955283+00:00


In [8]:
feedback_df.describe()
feedback_df[feedback_df["expert_id"] == "expert_1"]
feedback_df.groupby("expert_id").size()

expert_id
expert_1    24
expert_2    24
dtype: int64

In [9]:
users = load_users()
users   # a dictionary

{'irinalzr1': {'email': 'irinalzr1@gmail.com',
  'password': '$2b$12$hKy6DoQg4Rd4uNXCL11mwedjxZadaMzhZT8F/gJ7SXK.JfczypDA6',
  'role': 'admin'},
 'expert_1': {'email': 'expert1@kuleuven.be',
  'password': '$2b$12$NyV1YOsB3Jva9sXHJTRkVuOux/5yJwGXawZqKH3SHnkAtJfB9Iu2i',
  'role': 'expert'},
 'expert_2': {'email': 'expert2@kuleuven.be',
  'password': '$2b$12$mb4Qr7/rWhUxxG/tHA9uz.pXaYWYiogW1L2bUDNzkDzei.GUDZiFm',
  'role': 'expert'}}

In [ ]:
import pandas as pd
pd.DataFrame(users).T   # .T flips rows/columns since the dict is naturally keyed by username

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv("GITHUB_TOKEN"))

In [14]:
import os
import io
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.getenv("GITHUB_TOKEN")

def fetch_csv_from_github(owner, repo, branch, path, fname, token):
    raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{path}/{fname}"
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    resp = requests.get(raw_url, headers=headers, timeout=30)
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.text))

dnn_df = fetch_csv_from_github("margaridamascarenhas", "DAM_Forecast_V4", "main", "Forecast", "DNN_forecasts_10AM.csv", TOKEN)
qr_df = fetch_csv_from_github("margaridamascarenhas", "DAM_Forecast_V4", "main", "Forecast", "QR_forecasts_10AM.csv", TOKEN)
data_df = fetch_csv_from_github("margaridamascarenhas", "DAM_Forecast_V4", "main", "datasets", "Data_BE_UTC.csv", TOKEN)

print("DNN:", dnn_df.shape, "latest:", dnn_df["DateTime"].max())
print("QR:", qr_df.shape, "latest:", qr_df["DateTime"].max())
print("Data:", data_df.shape, "latest:", data_df["Date"].max())

DNN: (1248, 3) latest: 2026-08-14 23:45:00
QR: (16128, 41) latest: 2026-08-14 23:45:00
Data: (30528, 9) latest: 2026-08-14 23:45:00
